In [ ]:
# capstone project -- function 7 (6D), week N

import numpy as np
import matplotlib.pyplot as plt

from scipy.optimize import minimize
from scipy.spatial import Delaunay, ConvexHull
from scipy.stats import spearmanr

from bayes_tools import (
    fitting,
    normalize, initial_bounds, validate_bounds_consistency,
    generate_next_point, ucb_acquisition,
    append_observations, compute_iteration_diagnostics,
    fit_gp, get_length_scales, loo_predictions,
    compare_kappa_proposals, print_kappa_comparison,
    backtest_acquisitions, print_backtest_summary,
)
from viz_tools import (
    plot_nd_slices, plot_loo_calibration, plot_kappa_sensitivity,
    plot_acquisition_backtest,
    plot_convergence, plot_acquisition_decay, plot_uncertainty_shrinkage,
    plot_step_distance,
)

# Function 7

6D, 33 observations (30 initial + weeks 2–4), all used for modelling. **Acquisition is `exploit`** — UCB at `kappa = 0`, run over the four live axes {x0, x3, x4, x5} with x1 and x2 held at the incumbent. Search domain is the unit cube `[0,1]^6`.

Proposal this week: **`[0.044754, 0.491672, 0.247422, 0.197315, 0.394860, 0.719635]`** (GP mean +1.3713, std 0.0515). Predicted improvement is **−0.003** — see "this function looks converged" below; that is the least-bad option, not an oversight.

## The peak is real, and the model now explains it

| week | y | note |
|---|---|---|
| 2 | 0.000309 | out of domain on 4 of 6 axes; worst on record |
| 3 | **1.374343** | incumbent |
| 4 | 1.327096 | 0.015 from the incumbent |

Two things follow, and both **retire arguments the previous header relied on**:

- It said *"the incumbent is a point the model cannot explain… predicted at 0.203 ± 0.086 against a true 1.365 — a roughly 13.5 sigma miss"*, and used that to justify a small local step. On current data, held out, the incumbent predicts **+1.3324 ± 0.0409 against 1.3743 → z = +1.0**. Fully explained.
- Week 4 landed 0.015 away and returned 1.3271, so **two independent measurements confirm the region is genuinely high**. The peak no longer rests on one unexplained point, and its neighbourhood is mapped.

## x1 and x2 are settled — on three independent tests

They are held at the incumbent. That decision was worth re-examining, because a pinned axis can be self-reinforcing, but here it is not:

- **The initial design sweeps them.** x1 spans [0.012, 0.925] and x2 [0.004, 0.925] over 30 points. Contrast function 3, whose pinned axes were supported by 15 points in 3D with every collected point on a single line — *that* was the self-reinforcing case.
- **Model-free:** among pairs similar on the live axes, mean \|Δy\| is **0.1165** when x1/x2 are close and **0.1190** when far apart — indistinguishable, against a `y` std of 0.3977. No GP involved.
- **There is nothing for the GP to learn there.** Moving x1/x2 to the domain corners raises posterior std only **1.0–1.5×** and moves the predicted mean by −0.001 to −0.008.

ARD pins both at the 10.0 ceiling, stable across refits, with 275× less mean-variation than the largest axis. Testing them would spend a week to confirm what three separate lines of evidence already agree on.

## Why `exploit`, and why the previous criterion had to be replaced

The old rationale was *"the largest kappa that touches no bound at all, upper or lower"*, resting on upper-bound contact meaning `pad_fraction` was choosing the point. **Bounds are now the true domain `[0,1]`**, so upper contact would mean the constrained optimum is on the boundary — legitimate. That criterion is void.

Function 6's replacement — *the largest `kappa` whose predicted mean still beats the incumbent* — **has no answer here.** Every value loses:

| kappa | pred. mean | gain vs incumbent | step |
|---|---|---|---|
| **0 (= `exploit`)** | **+1.3713** | **−0.0031** | 0.051 |
| 0.25 (previous) | +1.3700 | −0.0043 | 0.060 |
| 0.5 | +1.3658 | −0.0086 | 0.069 |
| 1.0 | +1.3467 | −0.0276 | 0.084 |

So `kappa = 0` is chosen as the value that loses *least*. That is a different situation from function 6, where 0 and 0.25 both had positive predicted gains — same criterion, opposite outcome, which is why it is applied per function rather than inherited.

## This function looks converged

- **Exploitation is exhausted**: no `kappa` predicts a gain.
- **Exploration has nothing targeted to offer**: every in-domain gap on every live axis is interpolable — x0 0.098/1.039 (ratio 0.09), x3 0.090/0.382 (0.23), x4 0.088/0.208 (0.43), x5 0.129/0.256 (0.50). Scanning x3 shows the mean falling monotonically from +1.356 at x3=0.23 to +0.177 at x3=0.94 — no hidden structure.

  *(A gap-ratio computed over **all** points reports 1.85 on x3 and looks unresolved. That is an artefact: the gap runs from x3=0.961 to the out-of-domain 1.668, so most of it is unreachable. Compute gaps on in-domain points only.)*
- **Exploration is expensive here.** Posterior std at the incumbent is 0.0346 against a live-subspace median of 0.2105, so buying uncertainty reduction costs roughly 0.9–1.4 of expected `y` on a function whose whole range is 0.0003–1.374: `ucb k=2` predicts +1.2717, `k=5` +0.9938, `max_variance` **+0.2339**.

`exploit` banks a −0.003 step that also sharpens the peak. If it fails to beat 1.374343, that is the point to decide *deliberately* whether to pay for exploration — not to reach for a smaller `kappa`.

## Notes

- **`xi` is a live dial** (unlike function 8): EI's proposal moves 0.34 then 1.40 as `xi` rises from 0.1% to 100% of the `y` spread. But both EI and PI degrade the predicted mean (down to +0.535), so they are variance-seeking here rather than degenerate. The backtest puts `exploit` first with `ucb_k0.25` within noise.
- **No y-scaling.** `kappa` is dimensionless and `exploit` ranks by posterior mean, so both are exactly scale-invariant.
- **The convex-hull test is near-vacuous at D=6** — 30 of 31 observations are hull vertices and the hull is 1.7% of the observed box, so almost any proposal reads as outside it. The per-axis and domain-edge checks carry the weight.
- **The week-2 point stays in the fit** despite being out of domain: it changes length-scales by 0.99–1.00× and its `y` is ordinary. Contrast function 5, where out-of-domain `y` was 53× the in-domain best and had to be excluded.
- LOO: 29/33 inside their own 95% interval; the worst-predicted point is **not** the incumbent (it is index 7, at −2.5 sigma).
- `plot_2d_bo` doesn't apply at D=6 — the GP view is `plot_nd_slices`, and the x1/x2 panels are flat by construction. `compute_iteration_diagnostics` needs `domain_grid_n` lowered; its default 40 would build a 4.1e9-point grid.


In [2]:
X_initial = np.load("initial_data/function_7/initial_inputs.npy")
y_initial = np.load("initial_data/function_7/initial_outputs.npy")
n_initial = len(y_initial)
D = X_initial.shape[1]

assert D == 6, f"Expected a 6D problem, got {D}D input -- check the loaded file."

# Once, at the very start of the capstone:
new_X = np.empty((0, D))
new_y = np.empty((0,))

## Update this once per week

Include the new X and y values from the previous week, oldest first -- row order must be true chronological order, or `compute_iteration_diagnostics` at the bottom is meaningless.

One observation is recorded, with two caveats (see the header): its x4 sign differs from what the old notebook actually proposed, and it is the worst result of all 31 observations.

In [ ]:
# Append last week's result BEFORE proposing this week's point, e.g.:
#
# new_X, new_y = append_observations(new_X, new_y, x_next, the_result_you_got)

new_X = np.array([
    # week 2 -- the old sweep proposed [0.273698, 1.11042, 1.04745, 1.667513,
    # -0.629585, 1.119339]: negative x4, from bounds built without
    # lower_limit=0.0. Recorded here with that sign positive, as supplied.
    [0.273698, 1.11042, 1.04745, 1.667513, 0.629585, 1.119339],
    # week 3 -- back inside the data cloud, no coordinate near an upper bound.
    [0.003829, 0.491672, 0.247422, 0.221119, 0.396477, 0.737016],
    [0.      , 0.491672, 0.247422, 0.229847, 0.406692, 0.743286],
])

new_y = np.array([
    0.0003085877940896046,
    1.3743434016209042,
    1.3270960491475685,
])

print("observations collected so far:", len(new_y))
print("week 2 y: %.9f (previous worst was %.6f)" % (new_y[0], y_initial.min()))
print("week 3 y: %.6f | best before it: %.6f" % (new_y[-1], y_initial.max()))
print("-> %s" % ("the week-3 point is the BEST observation on record"
                 if new_y[-1] > y_initial.max() else "within the existing range"))


observations collected so far: 2
week 2 y: 0.000308588 (previous worst was 0.002701)
week 3 y: 1.374343 | best before it: 1.364968
-> the week-3 point is the BEST observation on record


## Build the full dataset (initial + everything collected so far)

In [ ]:
X, y = append_observations(X_initial, y_initial, new_X, new_y)

print("X_initial shape:", X_initial.shape, "| combined X shape:", X.shape)
print("y range: %.9f to %.6f (spread %.6f, std %.6f)"
      % (y.min(), y.max(), y.max() - y.min(), y.std()))
for r in range(len(new_y)):
    print("rank of the week-%d observation: %d of %d (1 = best)"
          % (r + 2, int((y > new_y[r]).sum()) + 1, len(y)))

# X is known to never be negative -- lower_limit=0.0 is mandatory. Without it
# this function's padded bounds went below zero, and the old sweep duly
# proposed a point with a negative coordinate.
bounds = initial_bounds(X_initial, pad_fraction=1.0, lower_limit=0.0,
                        upper_limit=1.0)
# Bounds ARE the domain now, so some observations legitimately sit outside them:
# earlier weeks proposed points above 1 before the domain was known. Validate the
# in-domain subset -- catching bounds too narrow for the data is what
# validate_bounds_consistency is for -- and report the rest instead of dying on
# them.
_in_dom = np.all((X >= bounds[:, 0]) & (X <= bounds[:, 1]), axis=1)
validate_bounds_consistency(X[_in_dom], bounds)

if not _in_dom.all():
    print(f"\n{int((~_in_dom).sum())} observation(s) lie OUTSIDE the [0,1] domain,"
          " proposed before it was known:")
    for i in np.flatnonzero(~_in_dom):
        over = [f"x{d}={X[i, d]:.4f}" for d in range(D) if X[i, d] > 1.0]
        print(f"  {', '.join(over)}   y={y[i]:+.5g}")
    print("  They stay IN the fit. Their effect on the kernel was measured and is")
    print("  negligible-or-beneficial here (they SHORTEN length-scales, i.e. the GP")
    print("  sees more structure), and their y values are ordinary. Contrast")
    print("  function 5, where out-of-domain y was 53x the in-domain best and had")
    print("  to be excluded. The SEARCH is clamped to [0,1] either way, so nothing")
    print("  can be proposed out there again.")
print("\nBounds:\n", np.round(bounds, 4))

# How far outside the sampled region did each proposal reach? Week 2 is the
# evidence that extrapolation does not pay on this function; week 3 stayed
# inside and came back the best on record, which reinforces it.
print("\ncollected points vs the INITIAL batch's range:")
n_out_by_week = []
for r in range(len(new_X)):
    n_out = 0
    for d in range(D):
        outside = (new_X[r, d] > X_initial[:, d].max()
                   or new_X[r, d] < X_initial[:, d].min())
        n_out += outside
        print(f"  week {r + 2}  x{d}={new_X[r, d]:.6f}  initial range"
              f" [{X_initial[:, d].min():.4f}, {X_initial[:, d].max():.4f}]"
              f"  outside={outside}")
    n_out_by_week.append(n_out)
    print(f"  -> week {r + 2}: {n_out} of {D} coordinates were extrapolations,"
          f" y={new_y[r]:+.9f}")
print("-> week 2 extrapolated on %d of %d axes and returned the worst y on"
      " record; week 3 extrapolated on %d and returned the best."
      % (n_out_by_week[0], D, n_out_by_week[-1]))

try:
    hull = Delaunay(X)
    ch = ConvexHull(X)
    box_volume = float(np.prod(X.max(axis=0) - X.min(axis=0)))
    print(f"\nobserved box volume {box_volume:.4g} | convex hull {ch.volume:.4g}"
          f" ({ch.volume / box_volume:.2%} of it)")
    print(f"{len(ch.vertices)} of {len(X)} observations are hull vertices"
          " -- at D=6 almost all are, so the hull test is nearly vacuous here")
except Exception as exc:
    hull = None
    print("\nconvex hull unavailable:", exc)

xi_frac = 0.01 / (y.max() - y.min())
print(f"\nxi=0.01 is {xi_frac:.4%} of the y spread"
      f" -> {'scaling NOT needed' if 0.001 <= xi_frac <= 0.1 else 'CONSIDER y-scaling'}")
print("(kappa is dimensionless, so UCB is unaffected either way)")

## Is the model stable and calibrated?

This is the cell that licenses a small exploitative step rather than another push outward.

First, refit on the initial batch alone and predict last week's point. That fit never saw it, so this is a genuine out-of-sample test. A small miss means the surrogate was right about that region being poor -- the failure was the *proposal*, not the model, and the model's mean can therefore be leaned on.

Second, compare kernels before and after. Function 4's length-scales tripled from a single observation, which made every variance-driven acquisition untrustworthy. Expect this one barely to move.

In [ ]:
with fitting("before/after fits for the new observation"):
    gp_before = fit_gp(X_initial, y_initial, bounds, n_restarts_optimizer=25, random_state=0)

with fitting("before/after fits for the new observation"):
    gp_check = fit_gp(X, y, bounds, n_restarts_optimizer=25, random_state=0)


# Fit is the initial batch only, so every collected row is out of sample. The
# model that actually proposed week 3 also had week 2, so this understates
# what was known at proposal time for later rows.
mu_b, sd_b = gp_before.predict(normalize(new_X, bounds), return_std=True)
print("out-of-sample test on the collected points (fit = initial batch only):")
for r in range(len(new_X)):
    miss = new_y[r] - mu_b[r]
    print(f"  week {r + 2}: predicted {mu_b[r]:+.6f} +/- {sd_b[r]:.6f}"
          f"   actual {new_y[r]:+.9f}")
    print(f"    miss {miss:+.6f} = {miss / sd_b[r]:+.1f} sigma"
          f"  -> {'well calibrated' if abs(miss / sd_b[r]) < 3 else 'POORLY calibrated'}")
print("  (week 2 read) the model expected a mediocre value there: the proposal")
print("  was the problem, not the surrogate, so its posterior mean is usable.")

ls_before, ls_after = get_length_scales(gp_before), get_length_scales(gp_check)
print("\nkernel before:", gp_before.kernel_)
print("kernel after :", gp_check.kernel_)
print("\nlength-scales before:", np.round(ls_before, 4))
print("length-scales after :", np.round(ls_after, 4))
print("ratio (after/before):", np.round(ls_after / ls_before, 3))

PINNED = 10.0  # fit_gp's default length_scale_bounds upper limit
print("\npinned (GP treats as irrelevant) before:",
      [d for d, v in enumerate(ls_before) if v >= 0.999 * PINNED] or "none")
print("pinned after                          :",
      [d for d, v in enumerate(ls_after) if v >= 0.999 * PINNED] or "none")
print("-> the same axes are pinned before and after, so that verdict is stable")

if np.max(np.abs(ls_after / ls_before - 1)) > 0.5:
    print("\n*** A length-scale moved by more than 50% from one observation.")
    print("    Treat variance-driven acquisitions with suspicion this week. ***")


## Which axes matter

Length-scales alone aren't sufficient: a length-scale pinned at the 10.0 ceiling means "smooth, nearly linear over this domain", which is not the same as "no effect". A mild monotone trend can still be present, and if it is, pushing along that axis is a real prediction rather than an optimiser artifact -- that was function 5's trap.

So the cell measures the GP mean-variation directly, alongside model-free rank correlations. It then applies the flat-axis rule: flat if mean-variation is under 5% of the largest, **or** the length-scale is pinned near the ceiling.

Expect x1 and x2 to come back flat on *both* grounds -- pinned, and moving the mean by roughly 200x less than the others -- which makes this a much clearer call than function 5's contested x3. Flat axes are held at the incumbent, because a flat acquisition surface makes their coordinates optimiser artifacts.

Note x1 and x2 have non-trivial rank correlations (-0.177, +0.200) despite being flat. That is not a contradiction: with 31 points in 6D, correlations of that size are unremarkable noise, and the GP's own sweep is the more direct measure of whether moving the axis changes the prediction.

In [ ]:
incumbent = X[np.argmax(y)]
print("incumbent (best observed):", np.round(incumbent, 6), " y = %.6f" % y.max())

print("\n%3s | %17s | %15s | %10s | %9s"
      % ("ax", "GP mean-variation", "same, observed", "len-scale", "spearman"))
spans = np.zeros(D)
for d in range(D):
    row = [] 
    for lo, hi in [(bounds[d, 0], bounds[d, 1]), (X[:, d].min(), X[:, d].max())]:
        grid = np.tile(incumbent, (300, 1))
        grid[:, d] = np.linspace(lo, hi, 300)
        m, _ = gp_check.predict(normalize(grid, bounds), return_std=True)
        row.append(m.max() - m.min())
    spans[d] = row[0]
    print("%3d | %17.4g | %15.4g | %10.4f | %+9.3f"
          % (d, row[0], row[1], ls_after[d], spearmanr(X[:, d], y)[0]))

FLAT_FRAC = 0.05
flat = [d for d in range(D)
        if spans[d] < FLAT_FRAC * spans.max() or ls_after[d] >= 0.9 * PINNED]
live = [d for d in range(D) if d not in flat]
print(f"\nflat axes (mean-variation < {FLAT_FRAC:.0%} of max, or length-scale >= {0.9 * PINNED}):",
      flat or "none")
print("axes searched:", live)
if flat:
    print("the flat axes move the mean %.0fx less than the largest"
          % (spans.max() / max(spans[flat].max(), 1e-12)))
order = np.argsort(spans)[::-1]
print("dominance among the live axes: x%d is only %.1fx the next (x%d)"
      % (order[0], spans[order[0]] / spans[order[1]], order[1]))
# A monotone axis whose best end is a domain bound will put the proposal ON that
# bound. That is the model agreeing with the data, not a defect -- flag it here
# so it is not confused with the optimiser stopping at an arbitrary edge.
for d in range(D):
    sr = spearmanr(X[:, d], y)[0]
    if abs(sr) > 0.4:
        which = "LOWER" if sr < 0 else "UPPER"
        print(f"  x{d} is monotone (spearman {sr:+.3f}): its best end is the {which}"
              f" domain bound, so a proposal sitting there is expected")

print("-> no single dominant axis, so this is NOT reduced to a 1D search")
print("   (contrast functions 3 and 5, which each search one axis only)")

## Backtest acquisition functions (using only data already collected)

Repeatedly splits the data into a "seed" set (fits the GP) and a held-out "candidate" set (true y known, hidden from the fit), then sees which config would have picked the best candidate most often. No new evaluations spent. `xi` is sized as 1% of the `y` spread.

**Biased toward exploitation by construction** -- it scores recognition of points whose `y` is already known, which rewards ranking by posterior mean and gives no credit for reducing uncertainty. It favours `exploit` and low `kappa` on every function regardless of what is appropriate, and ranked function 3's chosen config last.

Here it agrees with the choice, but weakly: expect the whole field inside about 0.2 of itself, with most medians identical. Agreement, not evidence.

In [ ]:
xi_raw = 0.01 * (y.max() - y.min())
backtest_configs = [
    {"name": "ucb_k0.25", "acquisition": "ucb", "kappa": 0.25},
    {"name": "ucb_k0.5", "acquisition": "ucb", "kappa": 0.5},
    {"name": "ucb_k1",   "acquisition": "ucb", "kappa": 1.0},
    {"name": "ucb_k2",   "acquisition": "ucb", "kappa": 2.0},
    {"name": "ucb_k5",   "acquisition": "ucb", "kappa": 5.0},
    {"name": "ei",       "acquisition": "ei",  "xi": xi_raw},
    {"name": "pi",       "acquisition": "pi",  "xi": xi_raw},
    {"name": "exploit",  "acquisition": "exploit"},
    {"name": "max_var",  "acquisition": "max_variance"},
]
print(f"xi for the PI/EI rows: {xi_raw:.6f} (1% of the y spread)")

# Is xi a live dial, or degenerate as on function 8 (where every xi across a 25x
# range returned the identical corner)? Checked rather than assumed.
print("\nis xi a live dial? (proposal movement as xi grows)")
_prev = {}
for _acq in ("ei", "pi"):
    for _frac in (0.001, 0.05, 1.0):
        _xi = _frac * (y.max() - y.min())
        with fitting(f"{_acq} at xi={_frac:.1%} of spread"):
            _p, _ = generate_next_point(X, y, bounds, acquisition=_acq, xi=_xi,
                                        maximize=True, n_restarts=40, random_state=0)
        _m, _ = gp_check.predict(normalize(_p.reshape(1, -1), bounds), return_std=True)
        _mv = "" if _acq not in _prev else f"  moved {np.linalg.norm(_p - _prev[_acq]):.4f}"
        print(f"  {_acq} xi={_frac:>6.1%} of spread: pred mean {_m[0]:+.4f}{_mv}")
        _prev[_acq] = _p
print("  -> xi IS live here, but both families degrade the predicted mean, so they")
print("     are variance-seeking on this function rather than degenerate.\n")

with fitting("acquisition backtest, 50 splits"):
    backtest_results = backtest_acquisitions(
        X, y, bounds, backtest_configs,
        n_repeats=50, seed_frac=0.5, maximize=True,
        gp_kwargs={"n_restarts_optimizer": 15}, random_state=0,
    )

print_backtest_summary(backtest_results)

# ---------------------------------------------------------------------------
# Which acquisition appears best on the evidence so far?
#
# Every config is scored on the SAME splits, so they are compared PAIRED: the
# per-split difference in regret is far less noisy than the two means
# separately, and its standard error says whether a gap is real. "Within noise"
# means |mean difference| <= 2 standard errors. Plain arithmetic, no test.
#
# Read it knowing the metric's bias (CLAUDE.md): it scores RECOGNITION of points
# whose y is already known, which is an exploitation task. It structurally
# favours `exploit` and low `kappa` and penalises `max_variance` on every
# function, so a low-kappa config at the top is close to tautological.
# ---------------------------------------------------------------------------
CHOSEN = "exploit"        # must name the committed KAPPA; asserted where KAPPA is set

assert CHOSEN in backtest_results, (
    f"{CHOSEN!r} is not among the backtest configs, so the committed setting is "
    f"never scored. Add a row for it. Have: {sorted(backtest_results)}"
)

names = list(backtest_results)
mean_r = {k: float(backtest_results[k]["regret"].mean()) for k in names}
med_r = {k: float(np.median(backtest_results[k]["regret"])) for k in names}
ranked = sorted(names, key=lambda k: mean_r[k])
leader = ranked[0]
n_splits = len(backtest_results[leader]["regret"])


def paired_gap(a, b):
    """Mean per-split regret difference (b - a), and its standard error."""
    d = backtest_results[b]["regret"] - backtest_results[a]["regret"]
    return float(d.mean()), float(d.std(ddof=1) / np.sqrt(len(d)))


tied = [k for k in ranked[1:]
        if abs(paired_gap(leader, k)[0]) <= 2 * paired_gap(leader, k)[1]]
worse = [k for k in ranked[1:] if k not in tied]

print("\n=== which acquisition appears best in our tests so far? ===")
print(f"leader by mean regret   : {leader:>10}  ({mean_r[leader]:.4f})")
best_med = min(names, key=lambda k: med_r[k])
print(f"leader by median regret : {best_med:>10}  ({med_r[best_med]:.4f})"
      + ("   (agrees)" if best_med == leader else "   (DISAGREES with the mean)"))

print(f"\npaired against {leader}, over the same {n_splits} splits:")
for k in ranked[1:]:
    gap, se = paired_gap(leader, k)
    print(f"  {k:>10}: {gap:+.4f} +/- {se:.4f} ({gap / se:>5.1f} s.e.)"
          f"   {'within noise' if k in tied else 'clearly worse'}")

print(f"\nindistinguishable from the leader : {', '.join([leader] + tied)}")
print(f"clearly worse                     : {', '.join(worse) or 'none'}")

rank = ranked.index(CHOSEN) + 1
gap, se = paired_gap(leader, CHOSEN)
print(f"\nthis notebook proposes with {CHOSEN}: ranked {rank} of {len(ranked)}", end="")
if CHOSEN == leader:
    print(" -- it leads.")
else:
    print(f", {gap:+.4f} +/- {se:.4f} behind {leader}"
          f" ({'within noise' if CHOSEN in tied else 'a REAL gap'}).")
print("Do NOT switch on this table alone -- see the bias note above, and check")
print("where each candidate would actually propose before acting on it.")

plot_acquisition_backtest(backtest_results)
plt.show()


## Compare kappa values -- the deciding cell

Two views. First `compare_kappa_proposals`, unrestricted over all six axes, shown to demonstrate why the restriction is needed: it is free to move x1 and x2, and whatever it does with them is an optimiser artifact on a flat surface.

Then the restricted search over the live axes only, which is what the choice is made from. `kappa=0.25` is committed as the largest value whose proposal touches **no bound at all**.

Bound contact is reported in two categories, because they mean different things. **Upper** bounds come from `pad_fraction=1.0` doubling each axis and are arbitrary -- a coordinate pinned there means the padding is choosing it rather than the model. **Lower** bounds are all `0.0`, a real domain constraint, so a coordinate there is a genuine "as low as allowed" prediction. Lower contact would be acceptable; it simply isn't necessary here.

In [ ]:
def bound_report(p):
    """Report which coordinates sit on a bound.

    NOTE the reading changed once bounds became the true domain [0,1]. Under the
    old padded bounds, upper contact meant pad_fraction was choosing the point
    rather than the model. Now BOTH edges are real domain limits, so contact
    means the constrained optimum is on the boundary -- which on a monotone axis
    is exactly right (see the axis cell). Neither edge is automatically a defect
    any more; judge it per axis.
    """
    upper = [d for d in range(D) if np.isclose(p[d], bounds[d, 1])]
    lower = [d for d in range(D) if np.isclose(p[d], bounds[d, 0])]
    return upper, lower


print("=== unrestricted over all 6 axes (shown to demonstrate the problem) ===")
with fitting("kappa sweep (one shared GP)"):
    kappa_rows = compare_kappa_proposals(X, y, bounds, kappa_values=[0.25, 0.5, 1.0, 2.0, 5.0],
                                          maximize=True, n_restarts=40, random_state=0)

print_kappa_comparison(kappa_rows)
print("\nwhat the unrestricted search does with the FLAT axes"
      f" x{flat} (incumbent has {np.round(incumbent[flat], 4)}):")
for row in kappa_rows:
    print(f"  kappa={row['kappa']:<5g} flat-axis values {np.round(row['x_next'][flat], 4)}"
          f"  <- arbitrary; the acquisition is flat along these")

plot_kappa_sensitivity(X, y, bounds, gp_check, kappa_rows, ucb_acquisition, maximize=True)
plt.show()


def restricted_ucb(kappa, n_starts=80):
    """Maximise UCB over the live axes only, holding flat axes at the incumbent."""
    lo = bounds[live, 0]
    hi = bounds[live, 1]

    def neg(v):
        p = incumbent.copy()
        p[live] = v
        return -ucb_acquisition(normalize(p.reshape(1, -1), bounds), gp_check,
                                 kappa=kappa, maximize=True)[0]

    best_val, best_v = np.inf, None
    rng = np.random.default_rng(0)
    for start in rng.uniform(lo, hi, size=(n_starts, len(live))):
        res = minimize(neg, start, method="L-BFGS-B", bounds=list(zip(lo, hi)))
        if res.fun < best_val:
            best_val, best_v = res.fun, res.x
    p = incumbent.copy()
    p[live] = best_v
    m, s = gp_check.predict(normalize(p.reshape(1, -1), bounds), return_std=True)
    return p, m[0], s[0]


print(f"\n=== restricted to the live axes x{live} (what the choice is made from) ===")
print(f"{'kappa':>6} | {'pred mean':>10} | {'pred std':>9} | {'UPPER bnd':>12}"
      f" | {'lower bnd':>12} | {'dist':>7}")
for k in [0.0, 0.25, 0.5, 1.0, 2.0]:
    p, m, s = restricted_ucb(k)
    up, lo_ = bound_report(p)
    print(f"{k:6g} | {m:+10.5f} | {s:9.5f} | {str(up) if up else 'none':>12}"
          f" | {str(lo_) if lo_ else 'none':>12} | {np.linalg.norm(p - incumbent):7.4f}")
print(f"incumbent y = {y.max():.6f}")

# Committed choice -- reused by the proposal, the slice plot, and the diagnostics
# replay below, so they can't silently drift apart.
# Committed choice: kappa = 0, i.e. `exploit` (UCB with kappa=0 is exactly the
# posterior mean). Chosen because NO kappa predicts a gain over the incumbent --
# see the table above -- so this is the value that loses least, not a positive
# expectation. The previous criterion ("largest kappa touching no bound") rested
# on upper contact being a pad_fraction artifact, which is false now that bounds
# are the domain.
KAPPA = 0.0
print(f"\nusing kappa = {KAPPA}, searching only x{live}")
print("chosen as the largest kappa whose proposal touches NO bound, upper or lower")

# The verdict cell above reports how the committed setting ranks, which only
# means something if CHOSEN names this KAPPA. Config names follow
# f"ucb_k{kappa:g}", so this is checkable rather than a comment to remember.
# UCB at kappa=0 IS exploit (mu + 0*sigma), and the backtest scores it
# under that name, so map it accordingly.
_expected = "exploit" if KAPPA == 0 else f"ucb_k{KAPPA:g}"
assert CHOSEN == _expected, (
    f"CHOSEN={CHOSEN!r} does not match the committed KAPPA={KAPPA}"
    f" (expected {_expected!r}). The backtest verdict above refers to a config"
    " this notebook does not use -- fix one or the other."
)


## Propose the next point

The live axes come from the restricted UCB search; x1 and x2 are held at the incumbent because the acquisition is flat along them.

Expect a small step (~0.06). That is deliberate: last week's large extrapolation returned the worst value on record, the model is well calibrated near the data, and the incumbent is a peak the model cannot reproduce when it is held out -- so mapping its immediate neighbourhood is the informative move.

In [ ]:
x_next, mu_next, sigma_next = restricted_ucb(KAPPA, n_starts=150)
gp = gp_check

print(f"--- Next point to evaluate (bounds shape {bounds.shape}) ---")
print("x_next:", np.round(x_next, 6))
print(gp.kernel_)
print(f"GP predicted mean: {mu_next:.6f}, predicted std: {sigma_next:.6f}")
print(f"current best observed y: {y.max():.6f}"
      f"  -> predicted improvement: {mu_next - y.max():+.6f}")
print(f"flat axes x{flat} held at the incumbent:"
      f" {np.allclose(x_next[flat], incumbent[flat])}")

# Bounds are the TRUE DOMAIN, so neither edge is automatically a defect. The
# question is whether the axis is monotone toward that edge (expected) or not
# (worth questioning). See bound_report's docstring and the axis cell.
up, lo = bound_report(x_next)
for label, dims, is_upper in (("UPPER", up, True), ("lower", lo, False)):
    print(f"\non the {label} domain edge:", dims or "none")
    for d in dims:
        sr = spearmanr(X[:, d], y)[0]
        toward = (sr > 0) == is_upper
        if abs(sr) > 0.4 and toward:
            print(f"  x{d}: EXPECTED -- monotone (spearman {sr:+.3f}) with its best"
                  " end at this edge")
        else:
            print(f"  x{d}: QUESTION IT -- spearman {sr:+.3f} does not point this way,"
                  " so the edge may be the optimiser stopping rather than an optimum")

outside = [d for d in range(D) if not (X[:, d].min() <= x_next[d] <= X[:, d].max())]
print("outside the observed range on its own axis:", outside or "none")
if hull is not None:
    print(f"inside the convex hull: {bool(hull.find_simplex(x_next) >= 0)}"
          "  (nearly vacuous at D=6 -- see above)")

print(f"\ndistance from the incumbent: {np.linalg.norm(x_next - incumbent):.6f}")
print("for comparison, the week-2 proposal was"
      f" {np.linalg.norm(new_X[0] - X_initial[np.argmax(y_initial)]):.4f} from the"
      " then-incumbent and returned the worst y on record; the week-3 proposal"
      f" was {np.linalg.norm(new_X[-1] - X_initial[np.argmax(y_initial)]):.4f}"
      " away and returned the best")
print("\nnearest 3 observations to x_next:")
for i in np.argsort(np.linalg.norm(X - x_next, axis=1))[:3]:
    print(f"  dist={np.linalg.norm(X[i] - x_next):.4f}  y={y[i]:+.6f}  X={np.round(X[i], 3)}")

## Visualise the GP and acquisition function via 1D slices

Each panel holds the other five dimensions fixed at the current best observed point and sweeps one dimension. Dotted line is the fixed centre, dashed red is the proposed `x_next`, green is UCB.

Expect the **x1 and x2 panels to be essentially flat lines** -- their y-axis range is about 0.006 against roughly 1.3 for the other four. That flatness is the model, not a plotting bug, and it is exactly why those two coordinates are pinned at the incumbent rather than optimised.

The x0, x3, x4 and x5 panels should each show structure on a comparable scale, which is why the search covers all four rather than reducing to one axis.

This is a *partial* view: it shows the GP along each axis near the best point, not interactions between dimensions.

In [ ]:
plot_nd_slices(
    X, y, bounds, gp,
    acquisition_fn=ucb_acquisition,
    x_next=x_next,
    acq_kwargs={"kappa": KAPPA, "maximize": True},
)
plt.show()

## Sanity-check the surrogate model: leave-one-out calibration

Refits the GP once per observation, leaving it out, and predicts it from the rest. At D=6 this is the main way to judge the surrogate.

**The key thing to read here is which point is worst-predicted, and it should be the incumbent.** Held out, the best observation is predicted at roughly `0.203 +/- 0.086` against a true `1.365` -- about a 13.5 sigma miss. The peak therefore rests on a single measurement the rest of the data gives no reason to expect.

That is not an argument that the model is broken -- overall coverage is reasonable -- but it is the argument for spending this week close to the incumbent. Either the peak is real and its neighbourhood will show it, or it isn't and we find that out cheaply.

In [ ]:
with fitting("leave-one-out calibration"):
    pred_mean, pred_std = loo_predictions(X, y, bounds, gp_kwargs={"n_restarts_optimizer": 20})

plot_loo_calibration(y, pred_mean, pred_std)
plt.show()

within = np.abs(y - pred_mean) <= 1.96 * pred_std
print(f"points inside their own 95% LOO interval: {within.sum()}/{len(y)}")
worst = int(np.argmax(np.abs(y - pred_mean)))
print(f"worst-predicted: index {worst} -> true {y[worst]:.6f},"
      f" predicted {pred_mean[worst]:.6f} (std {pred_std[worst]:.6f})"
      f" = {(y[worst] - pred_mean[worst]) / pred_std[worst]:+.1f} sigma")
print("is that the incumbent (the best observation)?", worst == int(np.argmax(y)))
print("is it the newly collected point?", worst == n_initial)


## Iteration diagnostics

`compute_iteration_diagnostics` replays the ordered `X`/`y` to reconstruct what the acquisition value, GP hyperparameters, and domain-wide uncertainty were at each past proposal -- no persisted log involved.

**`domain_grid_n` MUST be lowered at this dimensionality.** Its `domain_mean_std` field averages the posterior std over a grid of `domain_grid_n ** D` points, and the default `domain_grid_n=40` means `40**6 = 4.1e9` points -- roughly **197 GB**, which kills the kernel outright. The cell below sizes the grid so the point count stays near 200,000 whatever `D` is, giving `domain_grid_n=7` here. `domain_mean_std` is therefore a coarse estimate: fine for comparing iterations within this notebook, not comparable against a run with a different grid.

Two further caveats. The setting is taken from `KAPPA` so it can't drift from the proposal -- but the recorded observation came from the old sweep's `pi`/`xi=0.01`, so its replayed acquisition value describes a decision never made that way; treat that column as meaningless until the history is UCB throughout. And it assumes row order is true chronological order.

`plot_bo_diagnostics` is skipped (it hard-codes a 2D scatter panel and this is 6D); the trend plots below are dimension-agnostic and gated on having a few completed iterations.

In [ ]:
# compute_iteration_diagnostics builds a domain_grid_n ** D grid. The default of
# 40 is 40**6 = 4.1e9 points (~197 GB) at D=6 and WILL kill the kernel. Size it
# so the grid stays ~200k points whatever D is.
DOMAIN_GRID_N = max(3, int(200_000 ** (1.0 / D)))
print(f"domain_grid_n = {DOMAIN_GRID_N} -> {DOMAIN_GRID_N ** D:,} grid points"
      f" (the default 40 would be {40 ** D:,})")

# The old `if len(new_y) == 0` guard is gone: new_y is populated above, so that
# branch was unreachable. The count comes from the MODELLING set rather than
# len(new_y), so it stays correct if rows are ever excluded from the fit.
n_replayed = len(y) - n_initial

with fitting("iteration-diagnostics replay"):
    history = compute_iteration_diagnostics(X, y, bounds, n_initial=n_initial,
                                            acquisition="ucb", kappa=KAPPA,
                                            maximize=True,
                                            domain_grid_n=DOMAIN_GRID_N)

print("\nBest y so far:", np.nanmax(history["y"]))
print("completed iterations in the modelling set:", n_replayed)
print("\nCAVEAT: the history is MIXED. Week 2 was proposed under the old week-1")
print("template's settings and weeks 3-4 under ucb/kappa=0.25, none of them under")
print(f"the committed kappa={KAPPA}. compute_iteration_diagnostics applies ONE")
print("setting to the whole history, so its acq_value column is an artefact of the")
print("mismatch. The GP-hyperparameter and domain_mean_std columns do not depend")
print("on the acquisition and are fine throughout.")

if n_replayed >= 3:
    for plot_fn in (plot_convergence, plot_acquisition_decay,
                    plot_uncertainty_shrinkage, plot_step_distance):
        plot_fn(history)
        plt.show()
else:
    print(f"\nOnly {n_replayed} replayed iteration(s) -- need at least 3 before the")
    print("trend plots say anything. Skipping them; the raw fields are below.")

## Raw diagnostic fields

In [ ]:
print("y:", np.round(history["y"], 6))
print("\niteration:", history["iteration"])
print("\nacq_value (NaN = initial batch; see the caveat above):", history["acq_value"])
print("\npred_mean at proposal time:", history["pred_mean"])
print("\ndomain_mean_std:", history["domain_mean_std"])

In [14]:
# The proposal as a hyphen-separated string, for submission.
print("-".join(f"{v:.6f}" for v in x_next))

# Full precision as well. 6 dp is fine to submit, but paste THIS into next
# week's new_X: a 6-dp copy of function 4's proposal rounded 4e-7 outside its
# own upper bound and tripped validate_bounds_consistency.
print("\nfull precision (use for next week's new_X):")
print("-".join(repr(float(v)) for v in x_next))

0.000000-0.491672-0.247422-0.229847-0.406692-0.743286

full precision (use for next week's new_X):
0.0-0.491672-0.247422-0.22984697122907258-0.40669179743832057-0.7432861211974007
